# JBrowseR in Google Colab

[JBrowseR](https://github.com/GMOD/JBrowseR) renders the GPU-accelerated [JBrowse 2](https://jbrowse.org/jb2/) genome browser from R. This notebook runs it on Colab's **R runtime** (Runtime → Change runtime type → R).


## Install

Installs JBrowseR from GitHub. The package ships a prebuilt JavaScript bundle, so there is no JavaScript toolchain to set up.

In [ ]:
if (!requireNamespace("JBrowseR", quietly = TRUE)) {
  install.packages("remotes")
  remotes::install_github("GMOD/JBrowseR")
}

## A helper to show a browser inline

Colab renders R htmlwidgets most reliably inside a sandboxed iframe, so this helper saves a widget to self-contained HTML and embeds it.

In [ ]:
library(JBrowseR)
library(htmlwidgets)
library(IRdisplay)

show_browser <- function(widget, height = 480) {
  f <- tempfile(fileext = ".html")
  saveWidget(widget, f, selfcontained = TRUE)
  html <- paste(readLines(f, warn = FALSE), collapse = "\n")
  srcdoc <- gsub('"', '&quot;', html, fixed = TRUE)
  display_html(sprintf(
    '<iframe srcdoc="%s" width="100%%" height="%dpx" style="border:none;"></iframe>',
    srcdoc, height
  ))
}

## One line: a whole human genome browser

Name a hosted genome and JBrowseR loads the assembly, reference-name aliases, cytobands, and gene-name search. `location` accepts a gene symbol.

In [ ]:
show_browser(
  JBrowseR(
    "hg38",
    tracks = tracks(track(
      "https://s3.amazonaws.com/jbrowse.org/genomes/GRCh38/ncbi_refseq/GCA_000001405.15_GRCh38_full_analysis_set.refseq_annotation.sorted.gff.gz",
      name = "NCBI RefSeq Genes"
    )),
    location = "BRCA1"
  )
)

## Alignments

Add a track by URL — the track type and index files are inferred from the extension. Here, NA12878 exome reads over BRCA1.

In [ ]:
show_browser(
  JBrowseR(
    "hg38",
    tracks = tracks(track(
      "https://jbrowse.org/genomes/GRCh38/alignments/NA12878/NA12878.alt_bwamem_GRCh38DH.20150826.CEU.exome.cram",
      name = "NA12878 Exome"
    )),
    location = "17:43,044,295..43,048,000"
  )
)

## Your R results, on the genome

`track_data_frame()` turns a data frame into a track — no files, no server. A `score` column makes it quantitative.

In [ ]:
peaks <- data.frame(
  chrom = "17",
  start = seq(43000000, 43120000, by = 12000),
  end   = seq(43000000, 43120000, by = 12000) + 4000,
  name  = paste0("peak", 1:11),
  score = round(runif(11, 5, 100))
)

show_browser(
  JBrowseR(
    "hg38",
    tracks = list(track_data_frame(peaks, "R_peaks")),
    location = "17:43,000,000..43,125,000"
  )
)

## Cancer structural variants

The SKBR3 breast-cancer cell line has a rearranged genome. PacBio long reads span its breakpoints, and Sniffles calls the structural variants.

In [ ]:
show_browser(
  JBrowseR(
    "hg19",
    tracks = tracks(
      track(
        "https://jbrowse.org/genomes/hg19/SKBR3/reads_lr_skbr3.fa_ngmlr-0.2.3_mapped.bam.sniffles1kb_auto_l8_s5_noalt.filtered.vcf.gz",
        name = "Sniffles SV calls"
      ),
      track(
        "https://s3.amazonaws.com/jbrowse.org/genomes/hg19/skbr3/reads_lr_skbr3.fa_ngmlr-0.2.3_mapped.down.bam",
        name = "SKBR3 PacBio long reads"
      )
    ),
    location = "17:37,686,000..37,730,000"
  )
)

## Next steps

- Custom genomes: build an assembly from a FASTA URL with `assembly()`.
- Full control: pass a JBrowse `config.json` with `JBrowseR(config = ...)`.
- Shiny: use `JBrowseROutput()` / `renderJBrowseR()` and read `input$selectedFeature`.

Docs: <https://gmod.github.io/JBrowseR/>